# Optimal sizing and location of a BESS for multiuse flexibility provision

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import datetime
import cvxpy as cp

In [2]:
df_network = pd.read_excel('Network_rural.xlsx')

In [3]:
node_list = pd.read_excel('Nodes.xlsx')

In [4]:
df_community = pd.read_csv('scenario_s2.csv')
df_community

,Unnamed: 0,Date_Hour_start,b1,date,hour,ClusterNob1,ClusterNob2,b2,ClusterNob3,b3,...,ClusterNob7,b7,ClusterNob8,b8,ClusterNob9,b9,ClusterNob10,b10,Sum,daynu
0,0,2019-05-18 00:00:00,4.252,2019-05-18,0,0,0,7.610,1,9,...,2,9.736,1,2.406,0,1.552,0,7.5,49.787,137
1,1,2019-05-18 01:00:00,0.870,2019-05-18,1,0,0,1.718,1,10,...,2,2.153,1,3.228,0,2.117,0,9.0,36.406,137
2,2,2019-05-18 02:00:00,3.302,2019-05-18,2,0,0,0.990,1,9,...,2,2.641,1,3.888,0,2.437,0,6.0,34.821,137
3,3,2019-05-18 03:00:00,4.814,2019-05-18,3,0,0,1.090,1,9,...,2,3.497,1,3.447,0,2.353,0,9.0,38.032,137
4,4,2019-05-18 04:00:00,1.772,2019-05-18,4,0,0,0.980,1,7,...,2,1.866,1,2.391,0,1.508,0,7.5,28.005,137
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
691,691,2019-08-10 19:00:00,0.424,2019-08-10,19,0,0,0.374,1,5,...,2,0.586,1,2.583,0,4.040,0,58.5,76.689,221
692,692,2019-08-10 20:00:00,1.190,2019-08-10,20,0,0,0.548,1,5,...,2,1.143,1,2.472,0,1.941,0,42.0,60.037,221
693,693,2019-08-10 21:00:00,0.824,2019-08-10,21,0,0,0.720,1,5,...,2,1.132,1,4.047,0,2.629,0,16.5,37.769,221
694,694,2019-08-10 22:00:00,1.468,2019-08-10,22,0,0,4.716,1,7,...,2,5.450,1,3.396,0,2.375,0,9.0,43.563,221


In [5]:
price_da = pd.read_csv('Day-ahead22.csv')
price_up = pd.read_csv('up2022.csv')
price_down = pd.read_csv('down2022.csv')
freq1 = pd.read_csv('jan-march22.csv')
freq2 = pd.read_csv('AprilJun22.csv')
freq3 = pd.read_csv('JulySep22.csv')
freq4 = pd.read_csv('OctDeC22.csv')
capacity_price = pd.read_csv('fcr2022.csv')

In [6]:
freq = [freq1, freq2, freq3,freq4]
df_freq = pd.concat(freq)

In [7]:
df_freq = df_freq.reset_index()

In [8]:
df_freq

,index,Alkuaika UTC,Lopetusaika UTC,Alkuaika UTC+02:00,Lopetusaika UTC+02:00,Taajuus - reaaliaikatieto
0,0,2021-12-31 22:01:00,2021-12-31 22:01:00,2022-01-01 00:01:00,2022-01-01 00:01:00,49.967
1,1,2021-12-31 22:04:00,2021-12-31 22:04:00,2022-01-01 00:04:00,2022-01-01 00:04:00,49.967
2,2,2021-12-31 22:07:00,2021-12-31 22:07:00,2022-01-01 00:07:00,2022-01-01 00:07:00,49.964
3,3,2021-12-31 22:10:00,2021-12-31 22:10:00,2022-01-01 00:10:00,2022-01-01 00:10:00,49.984
4,4,2021-12-31 22:13:00,2021-12-31 22:13:00,2022-01-01 00:13:00,2022-01-01 00:13:00,49.930
...,...,...,...,...,...,...
181572,44123,2022-12-31 21:46:00,2022-12-31 21:46:00,2022-12-31 23:46:00,2022-12-31 23:46:00,50.014
181573,44124,2022-12-31 21:49:00,2022-12-31 21:49:00,2022-12-31 23:49:00,2022-12-31 23:49:00,50.003
181574,44125,2022-12-31 21:52:00,2022-12-31 21:52:00,2022-12-31 23:52:00,2022-12-31 23:52:00,50.000
181575,44126,2022-12-31 21:55:00,2022-12-31 21:55:00,2022-12-31 23:55:00,2022-12-31 23:55:00,50.018


In [9]:
data = {'time': df_freq['Alkuaika UTC+02:00'],
        'frequency': df_freq['Taajuus - reaaliaikatieto']}

In [10]:
df = pd.DataFrame(data)

In [11]:
df['FC'] = 50 - df['frequency']

In [12]:
df.loc[df[df['FC']<-0.1].index,'FC'] = -0.1
df.loc[df[df['FC']>0.1].index,'FC'] = 0.1

In [13]:
df['time'] = pd.to_datetime(df['time'])

In [14]:
df['hour'] = df.time.dt.hour
df['daynu'] = df.time.dt.dayofyear - 1

In [15]:
df

,time,frequency,FC,hour,daynu
0,2022-01-01 00:01:00,49.967,0.033,0,0
1,2022-01-01 00:04:00,49.967,0.033,0,0
2,2022-01-01 00:07:00,49.964,0.036,0,0
3,2022-01-01 00:10:00,49.984,0.016,0,0
4,2022-01-01 00:13:00,49.930,0.070,0,0
...,...,...,...,...,...
181572,2022-12-31 23:46:00,50.014,-0.014,23,364
181573,2022-12-31 23:49:00,50.003,-0.003,23,364
181574,2022-12-31 23:52:00,50.000,0.000,23,364
181575,2022-12-31 23:55:00,50.018,-0.018,23,364


In [16]:
data1 = {'time': price_up['Alkuaika UTC+02:00'],
        'price_up': price_up['Ylössäätöhinta säätösähkömarkkinoilla'],
        'price_down': price_down['Down-regulation price in the Balancing energy market'],
        'price_da': price_da['Day-ahead Price [EUR/MWh]'],
        'price_fcr_cap' :  capacity_price['Taajuusohjattu käyttöreservi, tuntimarkkinahinnat']}

In [17]:
price = pd.DataFrame(data1)

In [18]:
price['time'] = pd.to_datetime(price['time'])

In [19]:
price['hour'] = price.time.dt.hour
price['daynu'] = price.time.dt.dayofyear - 1

In [20]:
price['price_da'] = price['price_da'].shift(-1)

In [21]:
price

,time,price_up,price_down,price_da,price_fcr_cap,hour,daynu
0,2022-01-01 00:00:00,46.55,29.76,46.60,10.10,0.0,0.0
1,2022-01-01 01:00:00,46.60,46.60,41.33,2.50,1.0,0.0
2,2022-01-01 02:00:00,41.33,41.33,42.18,2.50,2.0,0.0
3,2022-01-01 03:00:00,42.18,42.18,44.37,3.50,3.0,0.0
4,2022-01-01 04:00:00,44.37,44.37,37.67,3.00,4.0,0.0
...,...,...,...,...,...,...,...
8756,2022-12-31 20:00:00,15.08,15.08,15.08,100.00,20.0,364.0
8757,2022-12-31 21:00:00,11.57,1.34,11.57,60.00,21.0,364.0
8758,2022-12-31 22:00:00,14.89,-7.00,14.89,18.77,22.0,364.0
8759,2022-12-31 23:00:00,9.94,-7.00,9.94,26.76,23.0,364.0


In [22]:
day_number = df_community.drop_duplicates(subset='daynu', keep='first')['daynu']
day_number

0      137
24     138
48     158
72     159
96     160
120    165
144    166
168    169
192    170
216    176
240    181
264    183
288    184
312    188
336    190
360    195
384    196
408    197
432    202
456    203
480    204
504    207
528    208
552    210
576    211
600    216
624    217
648    218
672    221
Name: daynu, dtype: int64

In [23]:
selected_day = [137, 176, 197, 221]

In [24]:
load_df = df_community[df_community['daynu'].isin(selected_day)]
load_df = load_df.reset_index(drop=True)

In [25]:
df['time'] =  pd.to_datetime(df['time'])
df.set_index(df['time'],inplace = True)

In [26]:
df.drop(columns=['time','frequency','hour','daynu'], inplace=True)

In [27]:
up_df = df[df.FC > 0].resample('1H').mean()

In [28]:
down_df = df[df.FC <= 0].resample('1H').mean()

In [29]:
up_df.fillna(0, inplace=True)
down_df.fillna(0, inplace=True)

In [30]:
down_df['FC'] = -down_df['FC']

In [31]:
fc = pd.DataFrame()
fc['up'] = up_df['FC']
fc['down'] = down_df['FC']
fc.index = up_df.index

In [32]:
fc.index= pd.to_datetime(fc.index)
fc['hour'] = fc.index.hour
fc['daynu'] = fc.index.dayofyear - 1

In [33]:
fc.reset_index(inplace=True)

In [34]:
load_df.columns.values

array(['Unnamed: 0', 'Date_Hour_start', 'b1', 'date', 'hour',
       'ClusterNob1', 'ClusterNob2', 'b2', 'ClusterNob3', 'b3',
       'ClusterNob4', 'b4', 'ClusterNob5', 'b5', 'ClusterNob6', 'b6',
       'ClusterNob7', 'b7', 'ClusterNob8', 'b8', 'ClusterNob9', 'b9',
       'ClusterNob10', 'b10', 'Sum', 'daynu'], dtype=object)

In [35]:
final_df = load_df[['b1','b2','b3','b4','b5','b6','b7','b8','b9','b10','hour','daynu']]

In [36]:
l = fc[fc.daynu.isin(final_df.daynu)].reset_index(drop=True)[['up','down']]

In [37]:
final_df = pd.concat([final_df, l], axis=1)

In [38]:
ll = price[price.daynu.isin(final_df.daynu)].reset_index(drop=True)[['price_up','price_down','price_da','price_fcr_cap']]

In [39]:
final_df = pd.concat([final_df, ll], axis=1)

In [40]:
final_df[['price_up','price_down','price_da','price_fcr_cap']] = 0.1 * final_df[['price_up','price_down','price_da','price_fcr_cap']] 

In [41]:
def node_index(node):
    return(node_list[node_list['node']==node].index[0])

In [42]:
def node_name(n):
    return(node_list['node'][n])

In [43]:
bes_location = node_name(10)
bes_location

'b10'

In [44]:
import scipy.io
data2 = scipy.io.loadmat('data_hosna.mat')
data3 = scipy.io.loadmat('PulseP.mat')

In [45]:
cell_energy = data2['Energy']
cell_cycle = data2['cycles']
cell_power = data3['PulseP']/3

In [46]:
rated_cell_power = (cell_power[0,5])/(1000 * 50)
rated_cell_energy = (cell_energy[0][0])/(1000 * 50)

In [47]:
(rated_cell_energy/rated_cell_power)*23.5627 

85.0740668309731

In [688]:
#bes_energy = (800/50)*(cell_energy[0,4]/cell_energy[0,0])
#bes_power = (221.57351472885196/50)*(cell_power[4,5]/cell_power[0,5])
daily_capital_cost = 100 * (150 + 0.01*150)/(8*365) #150 e/kwh + 10% installment cost + need to pay it over two years
daily_mo_cost = 0.001
cycling_cost = 10000 / (6000-cell_cycle[0,0]) #(euro / kwh cycle)

In [689]:
hourly_max_cycle = rated_cell_power / (2 * rated_cell_energy)
hourly_max_cycle

0.15499499029500524

In [690]:
final_df['first_hour'] = 0
final_df.loc[final_df[final_df.hour == 0].index,'first_hour'] = 1

In [691]:
final_df

,b1,b2,b3,b4,b5,b6,b7,b8,b9,b10,hour,daynu,up,down,price_up,price_down,price_da,price_fcr_cap,first_hour
0,4.252,7.610,9,0.798,4.268,2.665,9.736,2.406,1.552,7.5,0,137,0.026800,0.037867,2.316,2.316,2.316,19.000,1
1,0.870,1.718,10,1.724,1.969,3.627,2.153,3.228,2.117,9.0,1,137,0.030857,0.048692,1.849,1.849,1.849,16.500,0
2,3.302,0.990,9,0.793,1.040,4.730,2.641,3.888,2.437,6.0,2,137,0.001000,0.034833,1.258,0.500,1.258,16.500,0
3,4.814,1.090,9,1.196,1.088,2.547,3.497,3.447,2.353,9.0,3,137,0.024667,0.049364,1.258,0.100,1.258,16.500,0
4,1.772,0.980,7,1.563,1.833,1.592,1.866,2.391,1.508,7.5,4,137,0.039105,0.007000,1.993,0.100,1.993,19.000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,0.424,0.374,5,1.870,1.261,2.051,0.586,2.583,4.040,58.5,19,221,0.019333,0.022875,22.003,22.003,22.003,5.000,0
92,1.190,0.548,5,0.639,0.728,4.376,1.143,2.472,1.941,42.0,20,221,0.017583,0.009750,2.029,2.029,2.029,4.186,0
93,0.824,0.720,5,1.190,0.517,5.210,1.132,4.047,2.629,16.5,21,221,0.062133,0.028800,1.836,-0.241,1.836,2.900,0
94,1.468,4.716,7,0.998,2.566,6.594,5.450,3.396,2.375,9.0,22,221,0.019800,0.037067,0.241,0.048,0.241,4.900,0


In [692]:
# parameters
node_number = len(node_list)
T = len(final_df)
partitions = 10
v_min = 0.95
v_max = 1.05
v_nom = 1
q_factor = np.tan(np.arccos(0.9))
delta_s = v_nom * df_network['IMAX'] / partitions
R = range(1,partitions+1)
const = []   
t = 0
soc_min = 0.05
soc_max = 0.95
eff = 0.9
node_number = len(node_list)
T = len(final_df)
partitions = 10
v_min = 0.95
v_max = 1.05
v_nom = 1
q_factor = np.tan(np.arccos(0.9))
delta_s = v_nom * df_network['IMAX'] / partitions
R = range(1,partitions+1)
const = []
obj = 0   
t = 0
soc_min = 0.05
soc_max = 0.95
eff = 0.9
bes_power_min = 0.47125480772458106          # obtained from stage 1 pu
soc_ini = 0.5
bes_max_energy = 16 #pu #transformer capacity

# energy_cycling = 2 * bes_energy * cycle(t)
# cell_cycling_cost * energy_cycling / 2

In [693]:
0.47125480772458106  * 50

23.562740386229052

In [694]:
obj_value = {}
opt_flex_up = {}
opt_flex_down = {}
opt_bes_energy = {}
opt_bes_power = {}
opt_power_ch_spot = {}
opt_power_dis_spot = {}
opt_power_fcr_up = {}
opt_power_fcr_down = {}
opt_cap_fcr = {}
opt_power_act_poc = {}
opt_power_react_poc = {}
total_power = {}

In [695]:
def load(n):
    return(np.array(final_df.get(node_list['node'][n], default=np.zeros(final_df.shape[0]))))

# Variables:

# power trading and bes-related variables
power_act_poc = cp.Variable(shape = (node_number,T))                        # active power of poc
power_react_poc = cp.Variable(shape = (node_number,T))                      # reactive power of poc
#flex_up = cp.Variable(shape = (node_number,T), nonneg = True)               # loacl network upward flexibility
#flex_down= cp.Variable(shape = (node_number,T), nonneg = True)              # local network downward flexibility
power_ch_spot = cp.Variable(shape = (node_number,T), nonneg = True)         # power traded in spot for charging bes
power_dis_spot = cp.Variable(shape = (node_number,T), nonneg = True)        # power traded in spot for discharging bes
cap_fcr = cp.Variable(shape = (T,), nonneg = True)                          # FCR capacity
power_fcr_up = cp.Variable(shape = (node_number,T), nonneg = True)          # FCR upward activation
power_fcr_down = cp.Variable(shape = (node_number,T), nonneg = True)        # FCR downward activation
bes_power = cp.Variable(nonneg = True)                                      # BES rated power
bes_energy = cp.Variable(nonneg = True)                                     # BES capacity
bes_soe = cp.Variable(shape = (T,), nonneg = True)                          # BES state of energy (SOE)
energy_cycle = cp.Variable(shape = (T,), nonneg = True)                     # BES cycle

# network related parameters
flow_act_pos = cp.Variable(shape = (len(df_network),T), nonneg = True)      # positive active power flow
flow_act_neg = cp.Variable(shape = (len(df_network),T), nonneg = True)      # negative active power flow
flow_react_pos = cp.Variable(shape = (len(df_network),T), nonneg = True)    # positive reactive power flow
flow_react_neg = cp.Variable(shape = (len(df_network),T), nonneg = True)    # negative reactive power flow
voltage_square = cp.Variable(shape = (node_number,T), nonneg = True)        # voltage square
current_square = cp.Variable(shape = (len(df_network),T), nonneg = True)    # current square
power_act_delta = np.zeros((partitions+1), dtype=type(cp.Variable()))
for i in R:
    power_act_delta[i] = cp.Variable(shape=(len(df_network),T), nonneg = True)
power_react_delta = np.zeros((partitions+1), dtype=type(cp.Variable()))
for i in R:
    power_react_delta[i] = cp.Variable(shape=(len(df_network),T), nonneg = True)
par1 = np.zeros((partitions+1), dtype=type(cp.Variable()))
for i in R:
    par1[i] = cp.Variable(shape=(len(df_network),T), nonneg = True)
par2 = np.zeros((partitions+1), dtype=type(cp.Variable()))
for i in R:
    par2[i] = cp.Variable(shape=(len(df_network),T), nonneg = True)

# constraints
for t in range(T):
    for n in range(node_number):

        # balance constraints for active power
        const += [power_act_poc[n,t] - (load(n)[t] ) / 50 + power_dis_spot[n,t]
        #+ flex_up[n,t] 
        + power_fcr_up[n,t] 
        -  power_ch_spot[n,t] 
        #- flex_down[n,t]
        - power_fcr_down[n,t]
        - (np.array([df_network.fr == node_name(n)]).astype(int) @ flow_act_pos)[0,t]
        + (np.array([df_network.fr == node_name(n)]).astype(int) @ flow_act_neg)[0,t]
        - ((np.array([df_network.fr == node_name(n)]).astype(int) 
        * np.array(df_network.RL)) @ current_square)[0,t]
        + (np.array([df_network.to == node_name(n)]).astype(int) @ flow_act_pos)[0,t]
        - (np.array([df_network.to == node_name(n)]).astype(int) @ flow_act_neg)[0,t] == 0]
        
        # balance constraints for reactive power
        const += [power_react_poc[n,t] - q_factor * (load(n)[t]) / 50
        - (np.array([df_network.fr == node_name(n)]).astype(int) @ flow_react_pos)[0,t]
        + (np.array([df_network.fr == node_name(n)]).astype(int) @ flow_react_neg)[0,t]
        - ((np.array([df_network.fr == node_name(n)]).astype(int) 
        * np.array(df_network.XL)) @ current_square)[0,t]
        + (np.array([df_network.to == node_name(n)]).astype(int) @ flow_react_pos)[0,t]
        - (np.array([df_network.to == node_name(n)]).astype(int) @ flow_react_neg)[0,t] == 0]
        
        # Voltage constraints
        const += [voltage_square[n,t] <= np.square(v_max)]
        const += [np.square(v_min) <= voltage_square[n,t]]
        const += [voltage_square[1,t] == np.square(v_nom)]

        # Constraints related to the poc node
        if n != 0:
            const += [power_act_poc[n,t] == 0]
            const += [power_react_poc[n,t] == 0]

        # Constraints related to the bes location node
        if n != node_index(bes_location):
            const += [power_fcr_up[n,t] == 0]
            const += [power_fcr_down[n,t] == 0]
            const += [power_ch_spot[n,t] == 0]
            const += [power_dis_spot[n,t] == 0]
            #const += [flex_down[n,t] == 0]
            #const += [flex_up[n,t] == 0]

    # Network constraints:
    for net in range(len(df_network)):
         
        # voltage and power relationship:
        const += [voltage_square[node_index(df_network.loc[net,'fr']),t] -
        voltage_square[node_index(df_network.loc[net,'to']),t] -
        (np.square(df_network.RL[net]) + np.square(df_network.XL[net])) * current_square[net,t] -
        2 * df_network.RL[net] * (flow_act_pos[net,t] - flow_act_neg[net,t]) -
        2 * df_network.XL[net] * (flow_react_pos[net,t] - flow_react_neg[net,t]) == 0]

        # active and reactive power flow constraints
        const += [flow_act_pos[net,t] + flow_act_neg[net,t] <= v_nom * df_network.IMAX[net]]
        const += [flow_react_pos[net,t] + flow_react_neg[net,t] <= v_nom * df_network.IMAX[net]]

        # voltage constraint
        const += [current_square[net,t] <= np.square(df_network.IMAX[net])]

        # piece-wise linearization-related constraints
            
        const += [np.square(v_nom) * current_square[net,t] == 
        sum(par1[:])[net,t] + sum(par2[:])[net,t]]

        const += [flow_act_pos[net,t] + flow_act_neg[net,t] ==
        sum(power_act_delta[:])[net,t]]

        const += [flow_react_pos[net,t] + flow_react_neg[net,t] ==
        sum(power_react_delta[:])[net,t]]

        for r in R:
            const += [par1[r][net,t] == (2*r-1) * delta_s[net] * power_act_delta[r][net,t]]
            const += [par2[r][net,t] == (2*r-1) * delta_s[net] * power_react_delta[r][net,t]]
            const += [power_act_delta[r][net,t] <= delta_s[net]]
            const += [power_react_delta[r][net,t] <= delta_s[net]]
    
    # bes-related, spot trading and fcr trading- related constraints
    const += [cap_fcr[t] + power_ch_spot[node_index(bes_location),t] <= bes_power]
    const += [cap_fcr[t] + power_dis_spot[node_index(bes_location),t] <= bes_power]
    const += [power_fcr_up[node_index(bes_location),t] == (final_df.loc[t,'up']/0.1) * cap_fcr[t]]
    const += [power_fcr_down[node_index(bes_location),t] == (final_df.loc[t,'down']/0.1) * cap_fcr[t]]
    const += [bes_soe[t] == final_df.loc[t,'first_hour'] * soc_ini * bes_energy
    + (1-final_df.loc[t,'first_hour']) * bes_soe[t-1] + eff * (power_ch_spot[node_index(bes_location),t]
    + power_fcr_down[node_index(bes_location),t])
    - (1/eff) * (power_dis_spot[node_index(bes_location),t] 
    + power_fcr_up[node_index(bes_location),t])] #consider soe at the end of each time slot
    const += [bes_soe[t] <= soc_max * bes_energy]
    const += [soc_min * bes_energy <= bes_soe[t]]

    ##********************For day-ahead market use
    #const +=[cap_fcr[t] == 0]
    if t != 0:
        const += [cap_fcr[t] <= 2 * bes_soe[t-1] * (soc_max - soc_min)]
        const += [cap_fcr[t] <= 2 * (bes_energy - bes_soe[t-1]) * (soc_max - soc_min)]
    
    # Constraints related to the cycles and linearization using Mccormick envelopes:

    const += [energy_cycle[t] == power_ch_spot[node_index(bes_location),t] 
    + power_fcr_down[node_index(bes_location),t] + power_dis_spot[node_index(bes_location),t] 
    + power_fcr_up[node_index(bes_location),t]]

const += [bes_energy <= bes_max_energy]
const += [bes_energy >= (bes_power_min / rated_cell_power) * rated_cell_energy]
const += [bes_power == (bes_energy / rated_cell_energy) * rated_cell_power]
const += [bes_power >= bes_power_min]


obj = daily_capital_cost * 50 * bes_energy + 0.25 * (daily_mo_cost * 50 * cp.sum(energy_cycle) 
+ cycling_cost * 0.5 * 50 * cp.sum(energy_cycle) 
- 50 * cp.sum(cp.multiply(np.array(final_df['price_fcr_cap']) , cap_fcr))
- 50 * cp.sum(cp.multiply(np.array(final_df['price_up']) , power_fcr_up[node_index(bes_location),:]))
+ 50 * cp.sum(cp.multiply(np.array(final_df['price_down']) , power_fcr_down[node_index(bes_location),:]))
+ 50 * cp.sum(cp.multiply(np.array(final_df['price_da']) , (power_ch_spot[node_index(bes_location),:]
- power_dis_spot[node_index(bes_location),:]))))

problem = cp.Problem(cp.Minimize(obj), const)

obj_value = problem.solve(verbose=True)
#opt_flex_up = 50 * flex_up.value
#opt_flex_down = 50 * flex_down.value
opt_bes_energy = 50 * bes_energy.value
opt_bes_power = 50 * bes_power.value
opt_power_ch_spot = 50 * power_ch_spot.value
opt_power_dis_spot = 50 * power_dis_spot.value
opt_power_fcr_up = 50 * power_fcr_up.value
opt_power_fcr_down = 50 * power_fcr_down.value
opt_cap_fcr = 50 * cap_fcr.value
opt_power_act_poc = 50 * power_act_poc.value
opt_power_react_poc = 50 * power_react_poc.value
total_power = 50 * energy_cycle.value


                                     CVXPY                                     
                                     v1.2.1                                    
(CVXPY) Apr 13 01:27:35 PM: Your problem has 50880 variables, 57118 constraints, and 0 parameters.
(CVXPY) Apr 13 01:27:42 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Apr 13 01:27:42 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Apr 13 01:27:42 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
(CVXPY) Apr 13 01:27:51 PM: Compiling problem (target solver=ECOS).
(CVXPY) Apr 13 01:27:51 PM: Reduction chain: Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuf

In [697]:
obj_value

-23792.05067087578

In [349]:

"""
m_opt_bes_energy = 50 * bes_energy.value
m_opt_bes_power = 50 * bes_power.value
m_opt_power_ch_spot = 50 * np.matrix(power_ch_spot.value)
m_opt_power_dis_spot = 50 * np.matrix(power_dis_spot.value)
m_opt_power_fcr_up = 50 * np.matrix(power_fcr_up.value)
m_opt_power_fcr_down = 50 * np.matrix(power_fcr_down.value)
m_opt_cap_fcr = 50 * np.matrix(cap_fcr.value)
m_opt_power_act_poc = 50 * np.matrix(power_act_poc.value)
m_opt_power_react_poc = 50 * np.matrix(power_react_poc.value)
m_total_power = 50 * np.matrix(energy_cycle.value)
one = obj_value

In [ ]:
#m_opt_bes_power 

381.0853996456514

In [350]:
"""
np.savetxt("m_opt_power_ch_spot.csv", m_opt_power_ch_spot, delimiter=",")
np.savetxt("m_opt_power_dis_spot.csv", m_opt_power_dis_spot, delimiter=",")
np.savetxt("m_total_power.csv", m_total_power, delimiter=",")

In [181]:
#final_df.to_csv('2022.csv')

In [291]:
#n_opt_bes_energy = 50 * bes_energy.value
#n_opt_bes_power = 50 * bes_power.value
n_opt_power_ch_spot = 50 * np.matrix(power_ch_spot.value)
n_opt_power_dis_spot = 50 * np.matrix(power_dis_spot.value)
n_opt_power_fcr_up = 50 * np.matrix(power_fcr_up.value)
n_opt_power_fcr_down = 50 * np.matrix(power_fcr_down.value)
n_opt_cap_fcr = 50 * np.matrix(cap_fcr.value)
n_opt_power_act_poc = 50 * np.matrix(power_act_poc.value)
n_opt_power_react_poc = 50 * np.matrix(power_react_poc.value)
n_total_power = 50 * np.matrix(energy_cycle.value)

In [293]:
np.savetxt("n_opt_power_ch_spot.csv", n_opt_power_ch_spot, delimiter=",")
np.savetxt("n_opt_power_dis_spot.csv", n_opt_power_dis_spot, delimiter=",")
np.savetxt("n_opt_power_fcr_up.csv", n_opt_power_fcr_up, delimiter=",")
np.savetxt("n_opt_power_fcr_down.csv", n_opt_power_fcr_down, delimiter=",")
np.savetxt("n_opt_cap_fcr.csv", n_opt_cap_fcr, delimiter=",")
np.savetxt("n_total_power.csv", n_total_power, delimiter=",")